# Nigeria DHS 2024 climate linkage
## December 2023 to May 2024 interviews

This notebook extends the December pilot to the full 2024 survey round. Each household receives rainfall and temperature summaries for the **three complete calendar months before its interview month**.

| Interview month | Climate window | Expected households | Expected cluster-month rows |
|---|---|---:|---:|
| December 2023 | September–November 2023 | 4,902 | 173 |
| January 2024 | October–December 2023 | 9,221 | 329 |
| February 2024 | November 2023–January 2024 | 8,595 | 314 |
| March 2024 | December 2023–February 2024 | 10,536 | 367 |
| April 2024 | January–March 2024 | 6,759 | 235 |
| May 2024 | February–April 2024 | 34 | 2 |
| **Total** | | **40,047** | **1,420** |

The May interviews use February–April climate, so your climate downloads end in April 2024. The notebook keeps every household from the 2024 round and preserves its existing WASH and survey-design columns.


## 1. Choose the files

Open this notebook in Colab and select **Runtime → Run all**. Use each file picker to select **one file** from any folder on your computer. Wait for the next prompt after each upload.

| Order | Select |
|---|---|
| 1 | `Nigeria_DHS_WASH_Households.csv.gz` from the harmonisation notebook |
| 2–9 | Eight CHIRPS v3 GeoTIFFs in order from `chirps-v3.0.2023.09.tif` through `chirps-v3.0.2024.04.tif`. A compressed `.tif.gz` is also accepted. |
| 10 | `ERA5_Data.zip` for September–November 2023 |
| 11 | `ERA5_Data_Dec.zip` for December 2023 |
| 12 | `ERA5_Data_Jan.zip` for January–April 2024 |

The selected file is saved under the name shown in the prompt, so match each CHIRPS file to the requested month. If you stop partway, rerun this cell to resume with files already uploaded. To replace a file, delete its saved copy from `/content/climate_2024_inputs` in Colab's **Files** panel and rerun from this cell. Use a session where you are authorised to process the DHS household and cluster data.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile, is_zipfile
import gzip
import re
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
from PIL import Image

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

MONTHS = pd.period_range("2023-09", "2024-04", freq="M")
INTERVIEW_MONTHS = pd.period_range("2023-12", "2024-05", freq="M")
EXPECTED_HOUSEHOLDS = [4902, 9221, 8595, 10536, 6759, 34]
EXPECTED_CLUSTER_MONTHS = [173, 329, 314, 367, 235, 2]
ERA_ARCHIVES = {
    "ERA5_Data.zip": MONTHS[:3],
    "ERA5_Data_Dec.zip": MONTHS[3:4],
    "ERA5_Data_Jan.zip": MONTHS[4:],
}
RAIN_NAMES = [f"chirps-v3.0.{month.year}.{month.month:02d}.tif" for month in MONTHS]
INPUT_NAMES = ["Nigeria_DHS_WASH_Households.csv.gz", *RAIN_NAMES, *ERA_ARCHIVES]
KEY = ["survey_round", "hv001", "interview_month"]

INPUT_DIR = Path("/content/climate_2024_inputs" if files else "climate_2024_inputs")
OUTPUT_DIR = Path("/content/climate_2024_outputs" if files else "climate_2024_outputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

for number, name in enumerate(INPUT_NAMES, start=1):
    path = INPUT_DIR / name
    if not path.is_file() and files is not None:
        print(f"[{number}/{len(INPUT_NAMES)}] Choose {name}", flush=True)
        partial = path.with_suffix(path.suffix + ".part")
        try:
            files.upload_file(str(partial))
            if not partial.is_file() or partial.stat().st_size == 0:
                raise ValueError(f"No file uploaded for {name}. Rerun this cell to continue.")
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty input: {path}")
    print(f"[{number}/{len(INPUT_NAMES)}] Ready: {name}")

## 2. Select the 2024 round and check the cluster-month keys

`survey_round` identifies the survey release. `interview_month` identifies when a household was interviewed. A cluster interviewed in two different months needs two rows because its climate window changes. The expected counts below come from your harmonised export.


In [ ]:
households = pd.read_csv(INPUT_DIR / INPUT_NAMES[0], parse_dates=["interview_month"])
required = set(KEY + ["hv002", "gps_usable", "cluster_latitude", "cluster_longitude", "residence"])
if missing := required - set(households.columns):
    raise ValueError(f"Household export is missing columns: {sorted(missing)}")

round_2024 = households.loc[households.survey_round.eq(2024)].copy()
actual_counts = round_2024.groupby(round_2024.interview_month.dt.to_period("M")).size()
expected_counts = pd.Series(EXPECTED_HOUSEHOLDS, index=INTERVIEW_MONTHS)
assert actual_counts.equals(expected_counts), f"Unexpected 2024 household counts. Found:\n{actual_counts}"
assert round_2024[KEY + ["hv002"]].notna().all().all()
assert not round_2024.duplicated(["survey_round", "hv001", "hv002"]).any()

eligible = round_2024.gps_usable.astype(str).str.strip().str.lower().eq("true")
site_columns = ["cluster_latitude", "cluster_longitude", "residence"]
sites = round_2024.loc[eligible, KEY + site_columns].drop_duplicates().reset_index(drop=True)
assert not sites.duplicated(KEY).any(), "A cluster-month has inconsistent location or residence."
site_counts = sites.groupby(sites.interview_month.dt.to_period("M")).size()
expected_sites = pd.Series(EXPECTED_CLUSTER_MONTHS, index=INTERVIEW_MONTHS)
assert site_counts.equals(expected_sites) and eligible.all(), (
    f"Unexpected cluster-month counts or GPS coverage. Found:\n{site_counts}"
)
assert sites.residence.isin(["Urban", "Rural"]).all()
assert sites.cluster_latitude.between(4, 14).all()
assert sites.cluster_longitude.between(2, 15).all()
sites["buffer_km"] = sites.residence.map({"Urban": 2, "Rural": 5})
print(f"{len(round_2024):,} households and {len(sites):,} unique cluster-months.")

## 3. Read and check the climate grids

CHIRPS contains monthly rainfall totals in millimetres. ERA5-Land provides monthly mean air temperature at 2 metres. The code reads the actual ERA5 time coordinate, checks that every expected month is present once in its designated archive and converts Kelvin to Celsius when needed. A ZIP may contain one NetCDF for all its months or several NetCDF files.

If the archive contains GRIB, request **NetCDF** and select **Monthly averaged reanalysis** in the Copernicus download form. CHIRPS months are assigned by the upload prompt, so check the filename before selecting each raster.


In [ ]:
from dataclasses import dataclass

@dataclass
class Grid:
    values: np.ndarray  # Rows run north to south. Columns run west to east.
    west: float
    north: float
    dx: float
    dy: float

def read_rain(path):
    with path.open("rb") as handle:
        compressed = handle.read(2) == b"\x1f\x8b"
    open_grid = gzip.open if compressed else open
    with open_grid(path, "rb") as handle, Image.open(handle) as image:
        if image.format != "TIFF" or image.n_frames != 1:
            raise ValueError(f"Expected a single-band GeoTIFF: {path.name}")
        tags = image.tag_v2
        scale, tie = tags.get(33550), tags.get(33922)
        if scale is None or tie is None or 34264 in tags:
            raise ValueError(f"Expected a north-up GeoTIFF with a pixel scale and tie point: {path.name}")
        dx, dy = map(float, scale[:2])
        col, row, _, lon, lat, _ = map(float, tie[:6])
        keys = tags.get(34735, ())
        geokeys = {
            key: value for key, location, _, value in
            (keys[i:i + 4] for i in range(4, len(keys), 4)) if location == 0
        }
        if geokeys.get(2048, 4326) != 4326 or geokeys.get(1024, 2) != 2:
            raise ValueError(f"Expected a WGS84 longitude/latitude grid: {path.name}")
        west, north = lon - col * dx, lat + row * dy
        if geokeys.get(1025, 1) == 2:  # A point tie refers to a cell centre.
            west -= dx / 2
            north += dy / 2
        values = np.array(image, dtype=np.float64)
        nodata = tags.get(42113)
        if nodata is not None:
            values[values == float(nodata)] = np.nan
    if values.ndim != 2 or dx <= 0 or dy <= 0:
        raise ValueError(f"Unexpected CHIRPS grid: {path.name}")
    values[values < 0] = np.nan  # Negative rainfall indicates missing data.
    return Grid(values, west, north, dx, dy)

def attribute(variable, name, default=None):
    value = getattr(variable, name, default)
    return value.decode() if isinstance(value, bytes) else value

def read_numeric(variable):
    raw = np.ma.asarray(variable[:], dtype=np.float64).filled(np.nan)
    for name in ("_FillValue", "missing_value"):
        for invalid in np.atleast_1d(attribute(variable, name, np.nan)):
            raw[raw == invalid] = np.nan
    return raw * attribute(variable, "scale_factor", 1) + attribute(variable, "add_offset", 0)

def open_netcdf(path):
    with path.open("rb") as handle:
        signature = handle.read(8)
    if signature[:3] == b"CDF" and signature[3] in (1, 2):
        from scipy.io import netcdf_file
        return netcdf_file(path, mmap=False)
    if signature == b"\x89HDF\r\n\x1a\n" or signature[:3] == b"CDF":
        try:
            import netCDF4
        except ModuleNotFoundError:
            if files is None:
                raise RuntimeError("Install netCDF4 to read this NetCDF format.")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "netCDF4"])
            import netCDF4
        dataset = netCDF4.Dataset(path)
        dataset.set_auto_maskandscale(False)  # read_numeric applies it once.
        return dataset
    raise ValueError(f"{path.name} is not a supported NetCDF file. Request NetCDF in CDS.")

def read_months(variable):
    units = attribute(variable, "units", "")
    match = re.fullmatch(r"(seconds?|minutes?|hours?|days?) since (.+)", units, flags=re.I)
    calendar = attribute(variable, "calendar", "standard")
    if match is None or calendar not in ("standard", "gregorian", "proleptic_gregorian"):
        raise ValueError(f"Unsupported ERA5 time metadata: {units!r}, {calendar!r}")
    unit = {"second": "s", "minute": "min", "hour": "h", "day": "D"}[match[1].lower().rstrip("s")]
    dates = pd.to_datetime(
        pd.Timestamp(match[2]) + pd.to_timedelta(read_numeric(variable), unit=unit),
        utc=True,
    ).tz_convert(None)
    return dates.to_period("M")

def temperature_grids(path):
    with open_netcdf(path) as nc:
        v = nc.variables
        name = next((key for key in ("t2m", "2m_temperature", "tas") if key in v), None)
        if name is None:
            raise ValueError(f"No 2m air temperature variable in {path.name}.")
        temp = v[name]
        dims = list(temp.dimensions)
        time = next((key for key in ("valid_time", "time") if key in v and len(v[key].dimensions) == 1), None)
        lat = next((key for key in ("latitude", "lat") if key in v and len(v[key].dimensions) == 1), None)
        lon = next((key for key in ("longitude", "lon") if key in v and len(v[key].dimensions) == 1), None)
        if None in (time, lat, lon):
            raise ValueError("Expected one-dimensional time, latitude and longitude coordinates.")
        wanted = [v[key].dimensions[0] for key in (time, lat, lon)]
        if len(set(wanted)) != 3 or not set(wanted) <= set(dims):
            raise ValueError(f"Unexpected temperature dimensions: {dims}")
        values = read_numeric(temp)
        for axis in range(len(dims) - 1, -1, -1):
            if dims[axis] not in wanted:
                if values.shape[axis] != 1:
                    raise ValueError(f"Choose monthly averaged reanalysis. Extra dimension: {dims[axis]}")
                values = values.squeeze(axis=axis)
                dims.pop(axis)
        values = values.transpose([dims.index(key) for key in wanted])
        units = attribute(temp, "units", "").strip().lower()
        if units in ("k", "kelvin"):
            values -= 273.15
        elif units not in ("c", "degc", "degree_celsius", "degrees_celsius", "°c", "celsius"):
            raise ValueError(f"Review 2m temperature units: {units!r}")

        lat_values = read_numeric(v[lat])
        lon_values = (read_numeric(v[lon]) + 180) % 360 - 180
        iy, ix = np.argsort(lat_values)[::-1], np.argsort(lon_values)
        y, x = lat_values[iy], lon_values[ix]
        if len(x) < 2 or len(y) < 2:
            raise ValueError("Temperature grid needs at least two rows and columns.")
        dx, dy = x[1] - x[0], y[0] - y[1]
        if dx <= 0 or dy <= 0 or not np.allclose(np.diff(x), dx, atol=1e-5) or not np.allclose(-np.diff(y), dy, atol=1e-5):
            raise ValueError("Expected a regular ERA5-Land longitude/latitude grid.")
        dates = read_months(v[time])
        return [
            (month, Grid(values[i][np.ix_(iy, ix)], x[0] - dx / 2, y[0] + dy / 2, dx, dy))
            for i, month in enumerate(dates) if month in MONTHS
        ]

rain_grids = {month: read_rain(INPUT_DIR / name) for month, name in zip(MONTHS, RAIN_NAMES)}
temp_grids = {}
for archive_name, expected_months in ERA_ARCHIVES.items():
    archive_path = INPUT_DIR / archive_name
    archive_grids = []
    if is_zipfile(archive_path):
        with ZipFile(archive_path) as archive, TemporaryDirectory(dir=INPUT_DIR) as temp_dir:
            members = [
                member for member in archive.infolist()
                if not member.is_dir() and Path(member.filename).suffix.lower() in (".nc", ".nc4", ".netcdf")
            ]
            if not members:
                raise ValueError(f"{archive_name} has no NetCDF members. Request NetCDF in CDS.")
            for i, member in enumerate(members):
                target = Path(temp_dir) / f"member_{i}.nc"
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
                archive_grids.extend(temperature_grids(target))
    else:
        # CDS can deliver a single NetCDF even if it was saved with a .zip name.
        archive_grids = temperature_grids(archive_path)

    found = [month for month, grid in archive_grids]
    if sorted(found) != list(expected_months):
        raise ValueError(
            f"{archive_name}: expected {list(map(str, expected_months))}, "
            f"found {list(map(str, sorted(found)))} among the eight study months."
        )
    temp_grids.update(archive_grids)
    print(f"{archive_name}: {', '.join(map(str, sorted(found)))}")

print("Rainfall and temperature grids ready for:", ", ".join(map(str, MONTHS)))


## 4. Average the climate values inside each buffer

A buffer is a circle around the displaced cluster position. The code places sample points on a regular **200 m grid** inside the circle. Each point takes the climate value from the grid cell it falls in. Averaging those values approximates an area-weighted buffer mean, even when the climate grid is wider than the buffer. The method does not increase the underlying spatial resolution.

This run keeps the pilot’s **2 km urban** and **5 km rural** buffers. A monthly estimate is accepted only if at least **50%** of sample points contain valid grid values. The 50% threshold is the quality rule carried forward from the pilot. Retain the coverage fractions to review coastal and missing-data cases. There is no automatic fallback to a point value.

In [ ]:
SAMPLE_SPACING_M = 200
MIN_VALID_FRACTION = 0.50

def disc_offsets(radius_m):
    axis = np.arange(-radius_m, radius_m + SAMPLE_SPACING_M, SAMPLE_SPACING_M)
    east, north = np.meshgrid(axis, axis)
    inside = east**2 + north**2 <= radius_m**2
    return east[inside], north[inside]

OFFSETS = {km: disc_offsets(km * 1000) for km in (2, 5)}

def sample_locations(latitude, longitude, radius_km):
    east, north = OFFSETS[radius_km]
    bearing = np.arctan2(east, north)
    angular_distance = np.hypot(east, north) / 6_371_008.8
    y, x = np.deg2rad([latitude, longitude])
    lat = np.arcsin(
        np.sin(y) * np.cos(angular_distance) +
        np.cos(y) * np.sin(angular_distance) * np.cos(bearing)
    )
    lon = x + np.arctan2(
        np.sin(bearing) * np.sin(angular_distance) * np.cos(y),
        np.cos(angular_distance) - np.sin(y) * np.sin(lat),
    )
    return np.rad2deg(lat), np.rad2deg(lon)

def buffer_mean(grid, latitudes, longitudes):
    rows = np.floor((grid.north - latitudes) / grid.dy).astype(int)
    cols = np.floor((longitudes - grid.west) / grid.dx).astype(int)
    nrows, ncols = grid.values.shape
    inside = (rows >= 0) & (rows < nrows) & (cols >= 0) & (cols < ncols)
    values = grid.values[rows[inside], cols[inside]]
    valid = values[np.isfinite(values)]
    fraction = valid.size / latitudes.size
    return (float(valid.mean()) if fraction >= MIN_VALID_FRACTION else np.nan), fraction

monthly_rows = []
for site in sites.itertuples(index=False):
    latitudes, longitudes = sample_locations(site.cluster_latitude, site.cluster_longitude, site.buffer_km)
    interview = site.interview_month.to_period("M")
    for month in pd.period_range(interview - 3, interview - 1, freq="M"):
        rain, rain_fraction = buffer_mean(rain_grids[month], latitudes, longitudes)
        temp, temp_fraction = buffer_mean(temp_grids[month], latitudes, longitudes)
        monthly_rows.append({
            "survey_round": site.survey_round, "hv001": site.hv001,
            "interview_month": site.interview_month, "buffer_km": site.buffer_km,
            "climate_month": str(month), "days_in_month": month.days_in_month,
            "rain_mm": rain, "temp_c": temp,
            "rain_valid_fraction": rain_fraction, "temp_valid_fraction": temp_fraction,
        })
monthly = pd.DataFrame(monthly_rows)
assert len(monthly) == len(sites) * 3 and not monthly.duplicated(KEY + ["climate_month"]).any()
assert monthly.groupby(KEY).size().eq(3).all()
assert monthly.rain_mm.dropna().ge(0).all()
assert monthly.temp_c.dropna().between(-10, 60).all(), "Review ERA5-Land units or data range."
print("Extracted", len(monthly), "monthly cluster values for each variable.")

## 5. Make the three-month exposures and join them to households

Sum monthly rainfall. Weight temperature by the number of days in each month. A three-month exposure is left blank if any of its three monthly values is missing or falls below the coverage threshold.

The household merge is **many-to-one**. Many households can share a cluster-month, but the exposure table has only one row for that key.

For example, a household interviewed in April 2024 receives the January–March rainfall total and day-weighted mean temperature. February contributes 29 days in that window.


In [ ]:
monthly["temp_degree_days"] = monthly.temp_c * monthly.days_in_month
grouped = monthly.groupby(KEY, sort=True)
exposures = grouped.agg(
    buffer_km=("buffer_km", "first"),
    exposure_start=("climate_month", "min"),
    exposure_end=("climate_month", "max"),
    rain_months_valid=("rain_mm", "count"),
    temp_months_valid=("temp_c", "count"),
    min_rain_valid_fraction=("rain_valid_fraction", "min"),
    min_temp_valid_fraction=("temp_valid_fraction", "min"),
)
exposures["rain_mm_3m"] = grouped.rain_mm.sum(min_count=3)
exposures["temp_mean_c_3m"] = grouped.temp_degree_days.sum(min_count=3) / grouped.days_in_month.sum()
exposures = exposures.reset_index()
assert len(exposures) == len(sites) and not exposures.duplicated(KEY).any()
interview_periods = exposures.interview_month.dt.to_period("M")
assert exposures.exposure_start.eq((interview_periods - 3).astype(str)).all()
assert exposures.exposure_end.eq((interview_periods - 1).astype(str)).all()

joined = round_2024.merge(exposures, on=KEY, how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(round_2024) and joined._merge.eq("both").all()
assert not joined.duplicated(KEY + ["hv002"]).any()
joined = joined.drop(columns="_merge")
print(f"Joined {len(exposures):,} cluster-month rows to {len(joined):,} households.")

## 6. Save and review the results

Open Colab's **Files** sidebar and download the four files in `/content/climate_2024_outputs` before disconnecting. Start with `Nigeria_DHS_2024_Climate_QC.csv`. Confirm that all **40,047** households are retained, that there are **1,420** cluster-month rows and **4,260** monthly audit rows, and review the missing exposure counts. A household can match its cluster and still have missing climate data if a buffer fails the coverage check.

- **Households Climate** contains the original 2024 household rows with `rain_mm_3m` and `temp_mean_c_3m` attached.
- **Cluster Climate** contains one three-month summary per cluster and interview month.
- **Monthly Climate Audit** lists the three individual months and valid coverage fractions for each cluster-month.
- **Climate QC** shows household retention, climate windows and missingness for each interview month and overall.

If a month has missing exposures, use the monthly audit to check its climate layers and coverage fractions before using the household file for modelling.


In [ ]:
def qc_row(label, before, after, clusters, audit):
    return {
        "survey_round": 2024,
        "interview_month": label,
        "exposure_start": clusters.exposure_start.min(),
        "exposure_end": clusters.exposure_end.max(),
        "households_before": len(before),
        "households_after": len(after),
        "cluster_month_rows": len(clusters),
        "monthly_audit_rows": len(audit),
        "cluster_months_missing_rain": int(clusters.rain_mm_3m.isna().sum()),
        "cluster_months_missing_temperature": int(clusters.temp_mean_c_3m.isna().sum()),
        "households_missing_rain": int(after.rain_mm_3m.isna().sum()),
        "households_missing_temperature": int(after.temp_mean_c_3m.isna().sum()),
        "sampling_spacing_m": SAMPLE_SPACING_M,
        "min_valid_fraction": MIN_VALID_FRACTION,
    }

qc_rows = []
for month in INTERVIEW_MONTHS:
    date = month.to_timestamp()
    qc_rows.append(qc_row(
        str(month),
        round_2024.loc[round_2024.interview_month.eq(date)],
        joined.loc[joined.interview_month.eq(date)],
        exposures.loc[exposures.interview_month.eq(date)],
        monthly.loc[monthly.interview_month.eq(date)],
    ))
qc_rows.append(qc_row("TOTAL", round_2024, joined, exposures, monthly))
qc = pd.DataFrame(qc_rows)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exposures.to_csv(OUTPUT_DIR / "Nigeria_DHS_2024_Cluster_Climate.csv", index=False)
joined.to_csv(OUTPUT_DIR / "Nigeria_DHS_2024_Households_Climate.csv.gz", index=False, compression="gzip")
monthly.to_csv(OUTPUT_DIR / "Nigeria_DHS_2024_Monthly_Climate_Audit.csv", index=False)
qc.to_csv(OUTPUT_DIR / "Nigeria_DHS_2024_Climate_QC.csv", index=False)
display(qc)
print(f"Saved four files in {OUTPUT_DIR.resolve()}")


### Data sources

- [CHIRPS v3 Africa monthly GeoTIFFs](https://data.chc.ucsb.edu/products/CHIRPS/v3.0/monthly/africa/tifs/)
- [Copernicus ERA5-Land monthly averaged data](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-monthly-means)
- [DHS guidance on geospatial data and displacement](https://dhsprogram.com/Methodology/GPS-Data.cfm)